In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(44)

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

customers_df = pd.read_csv(
    project_root / "data" / "customers.csv"
)

loans_df = pd.read_csv(
    project_root / "data" / "loans.csv"
)

output_dir = project_root / "data" / "relational"
output_dir.mkdir(parents=True, exist_ok=True)

print("Customers:", len(customers_df))
print("Loans:", len(loans_df))

Customers: 10000
Loans: 15079


In [2]:
region_df = (
    customers_df[["region"]]
    .drop_duplicates()
    .sort_values("region")
    .reset_index(drop=True)
)

region_df.insert(
    0,
    "region_id",
    range(1, len(region_df) + 1)
)

region_df = region_df.rename(
    columns={"region": "region_name"}
)

region_df

,region_id,region_name
0,1,Blekinge län
1,2,Dalarnas län
2,3,Gotlands län
3,4,Gävleborgs län
4,5,Hallands län
5,6,Jämtlands län
6,7,Jönköpings län
7,8,Kalmar län
8,9,Kronobergs län
9,10,Norrbottens län


In [3]:
reference_date = pd.Timestamp("2026-09-10")

dates_of_birth = []

for age in customers_df["age"]:

    latest_birth_date = (
        reference_date
        - pd.DateOffset(years=int(age))
    )

    earliest_birth_date = (
        reference_date
        - pd.DateOffset(years=int(age) + 1)
        + pd.Timedelta(days=1)
    )

    number_of_days = (
        latest_birth_date - earliest_birth_date
    ).days

    random_days = np.random.randint(
        0,
        number_of_days + 1
    )

    date_of_birth = (
        earliest_birth_date
        + pd.Timedelta(days=random_days)
    )

    dates_of_birth.append(date_of_birth)

In [4]:
customer_df = customers_df[
    [
        "customer_id",
        "region"
    ]
].copy()

customer_df["date_of_birth"] = dates_of_birth

customer_df = customer_df.merge(
    region_df,
    left_on="region",
    right_on="region_name",
    how="left"
)

customer_df = customer_df[
    [
        "customer_id",
        "date_of_birth",
        "region_id"
    ]
]

customer_df.head(10)

,customer_id,date_of_birth,region_id
0,C00001,1985-06-14,12
1,C00002,1958-05-10,12
2,C00003,1970-03-03,2
3,C00004,1976-11-09,19
4,C00005,1998-12-16,21
5,C00006,1992-12-04,11
6,C00007,2006-05-08,12
7,C00008,1959-01-09,19
8,C00009,1978-02-09,12
9,C00010,1963-07-19,19


In [5]:
customer_employment_df = customers_df[
    [
        "customer_id",
        "employment_status",
        "monthly_income"
    ]
].copy()

customer_employment_df.insert(
    0,
    "employment_id",
    range(1, len(customer_employment_df) + 1)
)

customer_employment_df["valid_from"] = reference_date
customer_employment_df["valid_to"] = pd.NaT

customer_employment_df.head(10)

,employment_id,customer_id,employment_status,monthly_income,valid_from,valid_to
0,1,C00001,Temporary/Probation/Freelance,26400,2026-09-10,NaT
1,2,C00002,Retired,18900,2026-09-10,NaT
2,3,C00003,Permanent employment,30800,2026-09-10,NaT
3,4,C00004,Permanent employment,35700,2026-09-10,NaT
4,5,C00005,Permanent employment,45900,2026-09-10,NaT
5,6,C00006,Self-employed,29900,2026-09-10,NaT
6,7,C00007,Self-employed,57100,2026-09-10,NaT
7,8,C00008,Retired,24000,2026-09-10,NaT
8,9,C00009,Permanent employment,42600,2026-09-10,NaT
9,10,C00010,Permanent employment,40000,2026-09-10,NaT


In [6]:
loan_type_df = (
    loans_df[["loan_type"]]
    .drop_duplicates()
    .sort_values("loan_type")
    .reset_index(drop=True)
)

loan_type_df.insert(
    0,
    "loan_type_id",
    range(1, len(loan_type_df) + 1)
)

loan_type_df = loan_type_df.rename(
    columns={"loan_type": "loan_type_name"}
)

loan_type_df

,loan_type_id,loan_type_name
0,1,Car Loan
1,2,Credit Card
2,3,Mortgage
3,4,Personal Loan


In [7]:
loan_df = loans_df.merge(
    loan_type_df,
    left_on="loan_type",
    right_on="loan_type_name",
    how="left"
)

loan_df = loan_df[
    [
        "loan_id",
        "customer_id",
        "loan_type_id",
        "original_amount",
        "current_balance",
        "interest_rate",
        "monthly_payment",
        "late_payments",
        "default_flag"
    ]
].copy()

loan_df["default_flag"] = (
    loan_df["default_flag"].astype(bool)
)

loan_df.head(10)

,loan_id,customer_id,loan_type_id,original_amount,current_balance,interest_rate,monthly_payment,late_payments,default_flag
0,L000001,C00001,4,186000,106200,8.28,5980,0,False
1,L000002,C00002,3,470000,265600,2.92,1870,0,False
2,L000003,C00003,3,917000,824100,2.42,4150,0,False
3,L000004,C00004,1,93000,71800,6.70,3800,0,False
4,L000005,C00005,3,1592000,982400,3.50,6640,0,False
5,L000006,C00006,3,1138000,667100,3.14,5240,1,False
6,L000007,C00006,1,135000,72500,8.46,3170,0,False
7,L000008,C00007,2,21000,15500,15.50,1050,0,False
8,L000009,C00008,1,173000,84200,6.70,4260,0,False
9,L000010,C00009,3,1570000,1024700,2.67,6880,0,False


In [8]:
print("REGION:", len(region_df))
print("CUSTOMER:", len(customer_df))
print("CUSTOMER_EMPLOYMENT:", len(customer_employment_df))
print("LOAN_TYPE:", len(loan_type_df))
print("LOAN:", len(loan_df))

print("\nDuplicerade primary keys:")
print("region_id:", region_df["region_id"].duplicated().sum())
print("customer_id:", customer_df["customer_id"].duplicated().sum())
print(
    "employment_id:",
    customer_employment_df["employment_id"].duplicated().sum()
)
print(
    "loan_type_id:",
    loan_type_df["loan_type_id"].duplicated().sum()
)
print("loan_id:", loan_df["loan_id"].duplicated().sum())

print("\nSaknade foreign keys:")
print("customer.region_id:", customer_df["region_id"].isna().sum())
print(
    "employment.customer_id:",
    customer_employment_df["customer_id"].isna().sum()
)
print("loan.customer_id:", loan_df["customer_id"].isna().sum())
print("loan.loan_type_id:", loan_df["loan_type_id"].isna().sum())

REGION: 21
CUSTOMER: 10000
CUSTOMER_EMPLOYMENT: 10000
LOAN_TYPE: 4
LOAN: 15079

Duplicerade primary keys:
region_id: 0
customer_id: 0
employment_id: 0
loan_type_id: 0
loan_id: 0

Saknade foreign keys:
customer.region_id: 0
employment.customer_id: 0
loan.customer_id: 0
loan.loan_type_id: 0


In [9]:
region_df.to_csv(
    output_dir / "region.csv",
    index=False,
    encoding="utf-8"
)

customer_df.to_csv(
    output_dir / "customer.csv",
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d"
)

customer_employment_df.to_csv(
    output_dir / "customer_employment.csv",
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d"
)

loan_type_df.to_csv(
    output_dir / "loan_type.csv",
    index=False,
    encoding="utf-8"
)

loan_df.to_csv(
    output_dir / "loan.csv",
    index=False,
    encoding="utf-8"
)

print("Klart!")
print(output_dir)

Klart!
c:\Users\ELIYA\OneDrive\Dokument\nordic-credit-risk\data\relational
